# Data Loading, Storage, 

Accessing data is a necessary first step for using most of the tools in this book. I’m
going to be focused on data input and output using pandas, though there are numerous
tools in **other libraries** to help with reading and writing data in various formats.

Input and output typically falls into a few **main categories:** 
1. reading text files and other more efficient on-disk formats
2. loading data from databases
3. interacting with network sources like web APIs.

In [4]:
import numpy as np
import pandas as pd
np.random.seed(12345)
import matplotlib.pyplot as plt
plt.rc('figure', figsize=(10, 6))
np.set_printoptions(precision=4, suppress=True)

## Reading and Writing Data in Text Format
![](chan_folder/parsing_data_pandas.png)

In [5]:
#!cat examples/ex1.csv
!cd examples && type ex1.csv

a,b,c,d,message
1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo


In [6]:
df = pd.read_csv('examples/ex1.csv')
df

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [7]:
pd.read_table('examples/ex1.csv', sep=',')

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [8]:
#!cat examples/ex2.csv
!cd examples && type ex2.csv

1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo


A file will not always have a header row.

To read this file, you have a couple of options. You can allow pandas to assign default
column names, or you can specify names yourself:

In [11]:
print(pd.read_csv('examples/ex2.csv', header=None))
pd.read_csv('examples/ex2.csv', names=['a', 'b', 'c', 'd', 'message'])

   0   1   2   3      4
0  1   2   3   4  hello
1  5   6   7   8  world
2  9  10  11  12    foo


,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Suppose you wanted the `message` column to be the index of the returned DataFrame.
You can either indicate you want the column at index 4 or named `'message'` using
the `index_col` argument

In [12]:
names = ['a', 'b', 'c', 'd', 'message']
print(pd.read_csv('examples/ex2.csv', names=names, index_col='message'))
pd.read_csv('examples/ex2.csv', names=names, index_col=4)

         a   b   c   d
message               
hello    1   2   3   4
world    5   6   7   8
foo      9  10  11  12


,a,b,c,d
message,,,,
hello,1,2,3,4
world,5,6,7,8
foo,9,10,11,12


In [21]:
#ลองเอง
# ใช้ index_col โดยที่ไม่ตั้งชื่อคอลัมน์เอง
print(pd.read_csv('examples/ex2.csv', header=None))
print(pd.read_csv('examples/ex2.csv', header=None, index_col=4))

   0   1   2   3      4
0  1   2   3   4  hello
1  5   6   7   8  world
2  9  10  11  12    foo
       0   1   2   3
4                   
hello  1   2   3   4
world  5   6   7   8
foo    9  10  11  12


In the event that you want to form a hierarchical index from multiple columns, pass a
list of column numbers or names:

In [16]:
!cd examples && type csv_mindex.csv
parsed = pd.read_csv('examples/csv_mindex.csv',
                     index_col=['key1', 'key2'])
print(parsed)

parsed2 = pd.read_csv('examples/csv_mindex.csv',
                     index_col=['key2', 'key1'])
print(parsed2) #ไม่เวิร์ค เข้าใจว่าเพราะ key2 เดียวกันไม่อยู่ติดกันเป็นแถบ

parsed3 = pd.read_csv('examples/csv_mindex.csv',
                     index_col=[0, 'key2'])
print(parsed3)

key1,key2,value1,value2
one,a,1,2
one,b,3,4
one,c,5,6
one,d,7,8
two,a,9,10
two,b,11,12
two,c,13,14
two,d,15,16
           value1  value2
key1 key2                
one  a          1       2
     b          3       4
     c          5       6
     d          7       8
two  a          9      10
     b         11      12
     c         13      14
     d         15      16
           value1  value2
key2 key1                
a    one        1       2
b    one        3       4
c    one        5       6
d    one        7       8
a    two        9      10
b    two       11      12
c    two       13      14
d    two       15      16
           value1  value2
key1 key2                
one  a          1       2
     b          3       4
     c          5       6
     d          7       8
two  a          9      10
     b         11      12
     c         13      14
     d         15      16


In some cases, a table might not have a fixed delimiter, using whitespace or some
other pattern to separate fields. Consider a text file that looks like this:

In [23]:
!cd examples && type ex3.txt

list(open('examples/ex3.txt'))

            A         B         C
aaa -0.264438 -1.026059 -0.619500
bbb  0.927272  0.302904 -0.032399
ccc -0.264273 -0.386314 -0.217601
ddd -0.871858 -0.348382  1.100491


['            A         B         C\n',
 'aaa -0.264438 -1.026059 -0.619500\n',
 'bbb  0.927272  0.302904 -0.032399\n',
 'ccc -0.264273 -0.386314 -0.217601\n',
 'ddd -0.871858 -0.348382  1.100491\n']

the fields here are separated by a variable amount of whitespace. In these cases, you can pass a regular expression as a delimiter for `read_table`.

In [18]:
result = pd.read_table('examples/ex3.txt', sep='\s+')
result

,A,B,C
aaa,-0.264438,-1.026059,-0.619500
bbb,0.927272,0.302904,-0.032399
ccc,-0.264273,-0.386314,-0.217601
ddd,-0.871858,-0.348382,1.100491


#special_case

Because there was **one fewer column name** than the number of data rows, `read_table` infers that **the first column** should be the DataFrame’s index in this **special case**.

In [22]:
!cd examples && type ex4.csv
pd.read_csv('examples/ex4.csv', skiprows=[0, 2, 3])

# hey!
a,b,c,d,message
# just wanted to make things more difficult for you
# who reads CSV files with computers, anyway?
1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo


,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Handling missing values is an important and frequently nuanced part of the file parsing
process. Missing data is usually either not present (empty string) or marked by
some **sentinel** value. By default, pandas uses a set of commonly occurring sentinels,
such as `NA` and NULL:

In [25]:
!cd examples && type ex5.csv
result = pd.read_csv('examples/ex5.csv')
print(result)
pd.isnull(result)

something,a,b,c,d,message
one,1,2,3,4,NA
two,5,6,,8,world
three,9,10,11,12,foo
  something  a   b     c   d message
0       one  1   2   3.0   4     NaN
1       two  5   6   NaN   8   world
2     three  9  10  11.0  12     foo


,something,a,b,c,d,message
0,False,False,False,False,False,True
1,False,False,False,True,False,False
2,False,False,False,False,False,False


In [26]:
result = pd.read_csv('examples/ex5.csv', na_values=['NULL'])
result

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [28]:
#ลองเอง
result = pd.read_csv('examples/ex5.csv', na_values=['foo'])
result # 'NA' and '' are still treated as NaN

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,NaN


Different NA sentinels can be specified for each column in a dict:

In [29]:
sentinels = {'message': ['foo', 'NA'], 'something': ['two']}
pd.read_csv('examples/ex5.csv', na_values=sentinels) 
# 'foo' and 'two' become NaN

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,NaN,5,6,NaN,8,world
2,three,9,10,11.0,12,NaN


### Reading Text Files in Pieces

In [30]:
pd.options.display.max_rows = 10

In [31]:
result = pd.read_csv('examples/ex6.csv')
result

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q
...,...,...,...,...,...
9995,2.311896,-0.417070,-1.409599,-0.515821,L
9996,-0.479893,-0.650419,0.745152,-0.646038,E
9997,0.523331,0.787112,0.486066,1.093156,K
9998,-0.362559,0.598894,-1.843201,0.887292,G


In [32]:
pd.read_csv('examples/ex6.csv', nrows=5)

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q


#pieces

To read a file in pieces, specify a `chunksize` as a number of rows:

In [44]:
chunker = pd.read_csv('examples/ex6.csv', chunksize=1000)
chunker

In [84]:
chunker = pd.read_csv('examples/ex6.csv', chunksize=1000)

tot = pd.Series([], dtype=np.int32)
for piece in chunker:
    tot = tot.add(piece['key'].value_counts(), fill_value=0)

tot = tot.sort_values(ascending=False)

In [85]:
tot[:10]

E    368.0
X    364.0
L    346.0
O    343.0
Q    340.0
M    338.0
J    337.0
F    335.0
K    334.0
H    330.0
dtype: float64

In [43]:
#ลองเอง
t1 = pd.Series(['B','A','B','B'])
print(t1.value_counts())
t2 = pd.Series(['C','B','C','B'])
print(t2.value_counts(),'\n')

print(t1.value_counts().add(t2.value_counts()))
print(t1.value_counts().add(t2.value_counts(), fill_value=0), '\n')

tot1 = t1.value_counts().add(t2.value_counts(), fill_value=0)
print(tot1.sort_values(ascending=False))

B    3
A    1
dtype: int64
B    2
C    2
dtype: int64 

A    NaN
B    5.0
C    NaN
dtype: float64
A    1.0
B    5.0
C    2.0
dtype: float64 

B    5.0
C    2.0
A    1.0
dtype: float64


### Writing Data to Text Format

In [5]:
data = pd.read_csv('examples/ex5.csv')
data

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [8]:
data.to_csv('examples/out.csv')
!cd examples && type out.csv
print('\n')
import sys
data.to_csv(sys.stdout)
# สังเกตว่าจะมี comman เกินมาหนึ่งอันที่หน้าสุดของ column names

,something,a,b,c,d,message
0,one,1,2,3.0,4,
1,two,5,6,,8,world
2,three,9,10,11.0,12,foo


,something,a,b,c,d,message
0,one,1,2,3.0,4,
1,two,5,6,,8,world
2,three,9,10,11.0,12,foo


ลองเปรียบเทียบกับ #special_case ที่ ถ้าแถวบนสุดมี column น้อยกว่าแถวอื่น column แรกจะเป็น index

In [54]:
#ลองเอง
# ถ้าอ่าน output ออกมา ก็จะ treat index เหมือนเป็น data หรือว่าเพราะ แถวบน มี จ.น. column เท่าแถวอื่น?
pd.read_csv('examples/out.csv')

,Unnamed: 0,something,a,b,c,d,message
0,0,one,1,2,3.0,4,NaN
1,1,two,5,6,NaN,8,world
2,2,three,9,10,11.0,12,foo


In [65]:
#ลองเอง
# เอา comma ตัวแรกออก
# คราวนี้ 0,1,2 กลายเป็น index และ column 0 คือ one, two, three
del_one_comma = """something,a,b,c,d,message
0,one,1,2,3.0,4,
1,two,5,6,,8,world
2,three,9,10,11.0,12,foo"""
with open("examples/out_doc.csv","w") as f:
    f.write(del_one_comma)

data_doc=pd.read_csv('examples/out_doc.csv')
print(data_doc.iloc[:, 0])
data_doc

0      one
1      two
2    three
Name: something, dtype: object


,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [61]:
#ลองเอง
data_doc.values

array([['one', 1, 2, 3.0, 4, nan],
       ['two', 5, 6, nan, 8, 'world'],
       ['three', 9, 10, 11.0, 12, 'foo']], dtype=object)

In [52]:
import sys
data.to_csv(sys.stdout, sep='|')

|something|a|b|c|d|message
0|one|1|2|3.0|4|
1|two|5|6||8|world
2|three|9|10|11.0|12|foo


In [66]:
data.to_csv(sys.stdout, na_rep='NULL')

,something,a,b,c,d,message
0,one,1,2,3.0,4,NULL
1,two,5,6,NULL,8,world
2,three,9,10,11.0,12,foo


In [67]:
data.to_csv(sys.stdout, index=False, header=False)

one,1,2,3.0,4,
two,5,6,,8,world
three,9,10,11.0,12,foo


In [12]:
#small_yellow_duck 
# setting เดียวกับในโค้ด fb_auction
data.to_csv(sys.stdout, index=False, header=True)
print()

data.to_csv('examples/out_smyd.csv', index=False, header=True)
!cd examples && type out_smyd.csv

d = pd.read_csv('examples/out_smyd.csv')
d

something,a,b,c,d,message
one,1,2,3.0,4,
two,5,6,,8,world
three,9,10,11.0,12,foo

something,a,b,c,d,message
one,1,2,3.0,4,
two,5,6,,8,world
three,9,10,11.0,12,foo


,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [70]:
data.to_csv(sys.stdout, index=False, columns=['a', 'b', 'message'], na_rep='NULL')

a,b,message
1,2,NULL
5,6,world
9,10,foo


In [77]:
dates = pd.date_range('1/1/2000', periods=7)
ts = pd.Series(np.arange(7), index=dates)
ts.to_csv(sys.stdout)

print()
ts.to_csv(sys.stdout, header=False)


,0
2000-01-01,0
2000-01-02,1
2000-01-03,2
2000-01-04,3
2000-01-05,4
2000-01-06,5
2000-01-07,6

2000-01-01,0
2000-01-02,1
2000-01-03,2
2000-01-04,3
2000-01-05,4
2000-01-06,5
2000-01-07,6


### Working with Delimited Formats

In [86]:
!cd examples && type ex7.csv

"a","b","c"
"1","2","3"
"1","2","3"


In [8]:
import csv
f = open('examples/ex7.csv')

reader = csv.reader(f)

In [9]:
for line in reader:
    print(line)

['a', 'b', 'c']
['1', '2', '3']
['1', '2', '3']


In [10]:
with open('examples/ex7.csv') as f:
    lines = list(csv.reader(f)) #แต่ละ element of lines คือ a list containing data from a line
lines

[['a', 'b', 'c'], ['1', '2', '3'], ['1', '2', '3']]

In [11]:
#ลองเอง
with open('examples/ex7.csv') as f:
    my_lines = list(f) #แต่ละ element of lines คือ a list containing data from a line
my_lines

['"a","b","c"\n', '"1","2","3"\n', '"1","2","3"\n']

In [4]:
header, values = lines[0], lines[1:] 
# lines[0] เป็น 1-level list ['a', 'b', 'c']
# lines[1:] เป็น nested-list [['1', '2', '3'], ['1', '2', '3']] ซึ่งสามารถมองเป็น list of rows ได้

In [5]:
data_dict = {h: v for h, v in zip(header, zip(*values))} #list of rows เปลี่ยนเป็น list of columns แล้ว zip กับ list ของชื่อ column
data_dict

{'a': ('1', '1'), 'b': ('2', '2'), 'c': ('3', '3')}

In [7]:
#ลองเอง # ได้ผลลัพธ์เดียวกัน
data_dict_02 = dict(zip(header, zip(*values)))
data_dict_02

{'a': ('1', '1'), 'b': ('2', '2'), 'c': ('3', '3')}

In [92]:
class my_dialect(csv.Dialect):
    lineterminator = '\n'
    delimiter = ';'
    quotechar = '"'
    quoting = csv.QUOTE_MINIMAL

In [95]:
with open('examples/ex7.csv') as f:
    reader = csv.reader(f, dialect=my_dialect)

In [96]:
with open('examples/ex7.csv') as f:
    reader = csv.reader(f, delimiter='|')

In [97]:
with open('mydata.csv', 'w') as f:
    writer = csv.writer(f, dialect=my_dialect)
    writer.writerow(('one', 'two', 'three'))
    writer.writerow(('1', '2', '3'))
    writer.writerow(('4', '5', '6'))
    writer.writerow(('7', '8', '9'))

In [113]:
#ลองเอง #minimal #search

print("มี quote เกินมาใน field f")
!cd chan_folder && type notMinimal.csv
f = open('chan_folder/notMinimal.csv')
reader = csv.reader(f, quoting=csv.QUOTE_MINIMAL)
for row in reader:
    print(row)
    
print("\nมี quote เกิน & มีspace หน้า field นั้นๆ")
!cd chan_folder && type notMinimal_wsp.csv
f = open('chan_folder/notMinimal_wsp.csv')
reader = csv.reader(f, quoting=csv.QUOTE_MINIMAL)
for row in reader:
    print(row)
    
print("\nมี quote แค่ field which contain special characters")
!cd chan_folder && type minimal.csv
f = open('chan_folder/minimal.csv')
reader = csv.reader(f, quoting=csv.QUOTE_MINIMAL)
for row in reader:
    print(row)

มี quote เกินมาใน field f
"a, b, c",d
e,"f"
['a, b, c', 'd']
['e', 'f']

มี quote เกิน & มีspace หน้า field นั้นๆ
"a, b, c",d
e, "f"
['a, b, c', 'd']
['e', ' "f"']

มี quote แค่ field which contain special characters
"a, b, c",d
e,f
['a, b, c', 'd']
['e', 'f']


From official doc:
> `csv.reader(csvfile, /, dialect='excel', **fmtparams)`
<br>Return a reader object that will process lines from the given csvfile. A csvfile must be an iterable of strings, each in the reader’s defined csv format. A csvfile is most commonly a file-like object or list.

`f` เป็น an iterable of strings. ดูได้จากใน chapter 03
```
#iterable
with open(path) as f:
    lines = [x for x in f]

lines
```

From official doc
>`csv.QUOTE_MINIMAL` <br/>
Instructs writer objects to only quote those fields which contain special characters such as delimiter, quotechar, '\r', '\n' or any of the characters in lineterminator.

### JSON Data

In [115]:
obj = """
{"name": "Wes",
 "places_lived": ["United States", "Spain", "Germany"],
 "pet": null,
 "siblings": [{"name": "Scott", "age": 30, "pets": ["Zeus", "Zuko"]},
              {"name": "Katie", "age": 38,
               "pets": ["Sixes", "Stache", "Cisco"]}]
}
"""

JSON is very nearly valid Python code with the exception of its **null value** `null` and
some other nuances (such as disallowing trailing commas at the end of lists). The
basic types are objects (dicts), arrays (lists), strings, numbers, booleans, and nulls. All
of the keys in an object must be strings.

In [116]:
import json
result = json.loads(obj)
result

{'name': 'Wes',
 'places_lived': ['United States', 'Spain', 'Germany'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 30, 'pets': ['Zeus', 'Zuko']},
  {'name': 'Katie', 'age': 38, 'pets': ['Sixes', 'Stache', 'Cisco']}]}

In [117]:
asjson = json.dumps(result)

In [118]:
siblings = pd.DataFrame(result['siblings'], columns=['name', 'age']) #List of dicts...each dict becomes a row... see Table 5-1.
siblings

,name,age
0,Scott,30
1,Katie,38


In [119]:
#ลองเอง
siblings = pd.DataFrame(result['siblings'], columns=['name', 'age', 'pets'])
siblings

,name,age,pets
0,Scott,30,"[Zeus, Zuko]"
1,Katie,38,"[Sixes, Stache, Cisco]"


In [121]:
!cd examples && type example.json

[{"a": 1, "b": 2, "c": 3},
 {"a": 4, "b": 5, "c": 6},
 {"a": 7, "b": 8, "c": 9}]


The default options for pandas.read_json assume that each object in the JSON array
is a row in the table:

In [122]:
data = pd.read_json('examples/example.json')
data
# คล้ายๆเวลาป้อน list of dicts เข้า constructor - each dict become a row & union of dict keys becomes DF's column labels

,a,b,c
0,1,2,3
1,4,5,6
2,7,8,9


In [123]:
print(data.to_json())
# แต่ละ key-value pair แทน 1 column: key เป็น column name, value เป็น ค่าใน column ที่ถูกกำกับด้วย index
# น่าจะคล้าย เวลาใช้ dict of dicts - each inner dict become a column & keys are unioned to form the row index

print(data.to_json(orient='records'))
# กลับไปเป็น list of dicts เหมือนเนื้อหาใน file
# แต่ละ item(ซึ่งเป็น dict) แทน a row หรือเรียกอีกอย่างว่า record
# ค่าที่ print ออกมา ก็แสดงไปทีละ record สมดังที่ใช้ชื่อนี้ใน argument orient

{"a":{"0":1,"1":4,"2":7},"b":{"0":2,"1":5,"2":8},"c":{"0":3,"1":6,"2":9}}
[{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]


In [129]:
data.to_json('examples/out.json', orient='records')
!cd examples && type out.json 
# ดูเทียบ example.json

[{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]


### XML and HTML: Web Scraping

conda install lxml
pip install beautifulsoup4 html5lib

In [ ]:
tables = pd.read_html('examples/fdic_failed_bank_list.html')
len(tables)
failures = tables[0]
failures.head()

In [ ]:
close_timestamps = pd.to_datetime(failures['Closing Date'])
close_timestamps.dt.year.value_counts()

#### Parsing XML with lxml.objectify

<INDICATOR>
  <INDICATOR_SEQ>373889</INDICATOR_SEQ>
  <PARENT_SEQ></PARENT_SEQ>
  <AGENCY_NAME>Metro-North Railroad</AGENCY_NAME>
  <INDICATOR_NAME>Escalator Availability</INDICATOR_NAME>
  <DESCRIPTION>Percent of the time that escalators are operational
  systemwide. The availability rate is based on physical observations performed
  the morning of regular business days only. This is a new indicator the agency
  began reporting in 2009.</DESCRIPTION>
  <PERIOD_YEAR>2011</PERIOD_YEAR>
  <PERIOD_MONTH>12</PERIOD_MONTH>
  <CATEGORY>Service Indicators</CATEGORY>
  <FREQUENCY>M</FREQUENCY>
  <DESIRED_CHANGE>U</DESIRED_CHANGE>
  <INDICATOR_UNIT>%</INDICATOR_UNIT>
  <DECIMAL_PLACES>1</DECIMAL_PLACES>
  <YTD_TARGET>97.00</YTD_TARGET>
  <YTD_ACTUAL></YTD_ACTUAL>
  <MONTHLY_TARGET>97.00</MONTHLY_TARGET>
  <MONTHLY_ACTUAL></MONTHLY_ACTUAL>
</INDICATOR>

In [ ]:
from lxml import objectify

path = 'datasets/mta_perf/Performance_MNR.xml'
parsed = objectify.parse(open(path))
root = parsed.getroot()

In [ ]:
data = []

skip_fields = ['PARENT_SEQ', 'INDICATOR_SEQ',
               'DESIRED_CHANGE', 'DECIMAL_PLACES']

for elt in root.INDICATOR:
    el_data = {}
    for child in elt.getchildren():
        if child.tag in skip_fields:
            continue
        el_data[child.tag] = child.pyval
    data.append(el_data)

In [ ]:
perf = pd.DataFrame(data)
perf.head()

In [ ]:
from io import StringIO
tag = '<a href="http://www.google.com">Google</a>'
root = objectify.parse(StringIO(tag)).getroot()

In [ ]:
root
root.get('href')
root.text

## Binary Data Formats

In [ ]:
frame = pd.read_csv('examples/ex1.csv')
frame
frame.to_pickle('examples/frame_pickle')

In [ ]:
pd.read_pickle('examples/frame_pickle')

In [ ]:
!rm examples/frame_pickle

### Using HDF5 Format

In [ ]:
frame = pd.DataFrame({'a': np.random.randn(100)})
store = pd.HDFStore('mydata.h5')
store['obj1'] = frame
store['obj1_col'] = frame['a']
store

In [ ]:
store['obj1']

In [ ]:
store.put('obj2', frame, format='table')
store.select('obj2', where=['index >= 10 and index <= 15'])
store.close()

In [ ]:
frame.to_hdf('mydata.h5', 'obj3', format='table')
pd.read_hdf('mydata.h5', 'obj3', where=['index < 5'])

In [ ]:
os.remove('mydata.h5')

### Reading Microsoft Excel Files

In [ ]:
xlsx = pd.ExcelFile('examples/ex1.xlsx')

In [ ]:
pd.read_excel(xlsx, 'Sheet1')

In [ ]:
frame = pd.read_excel('examples/ex1.xlsx', 'Sheet1')
frame

In [ ]:
writer = pd.ExcelWriter('examples/ex2.xlsx')
frame.to_excel(writer, 'Sheet1')
writer.save()

In [ ]:
frame.to_excel('examples/ex2.xlsx')

In [ ]:
!rm examples/ex2.xlsx

## Interacting with Web APIs

In [ ]:
import requests
url = 'https://api.github.com/repos/pandas-dev/pandas/issues'
resp = requests.get(url)
resp

In [ ]:
data = resp.json()
data[0]['title']

In [ ]:
issues = pd.DataFrame(data, columns=['number', 'title',
                                     'labels', 'state'])
issues

## Interacting with Databases

In [ ]:
import sqlite3
query = """
CREATE TABLE test
(a VARCHAR(20), b VARCHAR(20),
 c REAL,        d INTEGER
);"""
con = sqlite3.connect('mydata.sqlite')
con.execute(query)
con.commit()

In [ ]:
data = [('Atlanta', 'Georgia', 1.25, 6),
        ('Tallahassee', 'Florida', 2.6, 3),
        ('Sacramento', 'California', 1.7, 5)]
stmt = "INSERT INTO test VALUES(?, ?, ?, ?)"
con.executemany(stmt, data)
con.commit()

In [ ]:
cursor = con.execute('select * from test')
rows = cursor.fetchall()
rows

In [ ]:
cursor.description
pd.DataFrame(rows, columns=[x[0] for x in cursor.description])

In [ ]:
import sqlalchemy as sqla
db = sqla.create_engine('sqlite:///mydata.sqlite')
pd.read_sql('select * from test', db)

In [ ]:
!rm mydata.sqlite

## Conclusion